# Homework 9 — Spark MLlib Churn Models

Name: Alex Devoid  
Course: ST 554

In this notebook I use Spark MLlib to model customer churn from Telco customer data. I read the data from a URL, create a Spark SQL DataFrame, and use Spark transformations to prepare the `label` and feature columns for modeling. I then split the data into training and test sets, fit three model classes with MLlib pipelines and cross-validation, and use Spark actions to summarize the data and compare the best models on the test set.

I keep the response binary and use area under the ROC curve, the AUC, as the main model metric.

In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from urllib.request import urlretrieve
from IPython.display import display

# import Spark SQL and MLlib
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, Imputer
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder


pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:0.4f}')

# random seed and file paths
RANDOM_STATE = 554
HW9_DIR = Path('/Users/alexdevoid/Documents/Stats/ST554-HW/HW9')
DATA_URL = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
DATA_PATH = HW9_DIR / 'Telco-Customer-Churn.csv'

# create Spark session
spark = SparkSession.builder.master('local[*]').appName('hw9_telco_churn').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

## 1. Read and Prepare the Data

I use the IBM Telco Customer Churn data. The response is `Churn` where `Yes` means the customer left. I save a local copy in the homework folder and then create a Spark SQL DataFrame for the modeling steps.

I convert `TotalCharges` to a numeric type and let the Spark pipeline impute the missing values later.

In [ ]:
# save a copy of the CSV
urlretrieve(DATA_URL, DATA_PATH)

# read the file, fix the numeric column with blank strings
telco_pdf = pd.read_csv(DATA_PATH)
telco_pdf['TotalCharges'] = pd.to_numeric(telco_pdf['TotalCharges'], errors='coerce')
telco_pdf['SeniorCitizen'] = telco_pdf['SeniorCitizen'].astype(float)

# create a Spark SQL DataFrame, then add the binary label
telco = spark.createDataFrame(telco_pdf)
telco = telco.withColumn('label', when(col('Churn') == 'Yes', 1.0).otherwise(0.0))

# count the rows, columns, and missing TotalCharges values before the split.
dataset_summary = pd.DataFrame(
    {
        'rows': [len(telco_pdf)],
        'original_columns': [telco_pdf.shape[1]],
        'missing_total_charges': [int(telco_pdf['TotalCharges'].isna().sum())],
        'spark_columns_with_label': [len(telco.columns)],
        
    }
)

# count churn outcomes in Spark, then sort with pandas
churn_summary = telco.groupBy('Churn').count().toPandas().sort_values('Churn').reset_index(drop=True)
# find the churn proportions.
churn_summary['proportion'] = churn_summary['count'] / churn_summary['count'].sum()

# keep a short list of columns for the sample preview table.
sample_cols = ['gender', 'SeniorCitizen', 'tenure', 'Contract', 'MonthlyCharges', 'TotalCharges', 'Churn', 'label']

# show the size summary and response proportions
display(dataset_summary)
display(churn_summary)
display(telco.select(sample_cols).limit(5).toPandas())

,rows,original_columns,missing_total_charges,spark_columns_with_label
0,7043,21,11,22


,Churn,count,proportion
0,No,5174,0.7346
1,Yes,1869,0.2654


,gender,SeniorCitizen,tenure,Contract,MonthlyCharges,TotalCharges,Churn,label
0,Female,0.0000,1,Month-to-month,29.8500,29.8500,No,0.0000
1,Male,0.0000,34,One year,56.9500,1889.5000,No,0.0000
2,Male,0.0000,2,Month-to-month,53.8500,108.1500,Yes,1.0000
3,Male,0.0000,45,One year,42.3000,1840.7500,No,0.0000
4,Female,0.0000,2,Month-to-month,70.7000,151.6500,Yes,1.0000


The data set has 7,043 customers. After converting `TotalCharges` to numeric, 11 rows still have missing values there, so I left that fix to an imputer inside the model pipelines. The response has 1,869 churners and 5,174 that had not churned.

## 2. Split the Data, Metric, and Model Ideas

I split the Spark DataFrame into training and test sets with `randomSplit`, using the same split for all models. To compare models I use AUC via Spark’s `BinaryClassificationEvaluator`. I am using AUC because the ROC curve compares true positive rate and false positive rate over different classification thresholds.

The three models are:

- `Logistic regression`: a model for a binary response that uses the logistic function to model the probability of success.
- `Decision tree`: a tree-based model that splits up the predictor space into regions and gives a prediction for each region.
- `Random forest`: a tree-based ensemble model that fits many trees on bootstrap samples and uses a random subset of predictors at each split before combining the trees into one final prediction.


In [ ]:
# split the Spark DataFrame into training and test sets with the fixed seed.
train, test = telco.randomSplit([0.75, 0.25], seed=RANDOM_STATE)
# cache splits 
train = train.cache()
test = test.cache()

# summarize split sizes and churn rates 
def split_summary_row(frame, label):

    return {
        'split': label,
        'rows': frame.count(),
        'churn_rate': frame.selectExpr('avg(label) as churn_rate').first()['churn_rate'],
    }

# stack summaries into a dataframe
split_summary = pd.DataFrame(
    [
        split_summary_row(train, 'training'),
        split_summary_row(test, 'test'),
    ]
)

# show the split sizes and churn rates
display(split_summary)

,split,rows,churn_rate
0,training,5309,0.2684
1,test,1734,0.2561


The split gives me 5,309 training rows and 1,734 test rows. The churn rates are `0.2684` in training and `0.2561` in test, so the two splits stay fairly close

## 3. Fit the Models

The logistic regression pipeline does the most preprocessing. It imputes `TotalCharges`, encodes the categorical columns, builds the feature vector, and then standardizes the features before fitting the model. The tree pipelines use the same starting transformations, but not the scaling step.


I tune each model with cross-validation on the training data and then compare the best cross-validated AUC values before moving to the test set.

In [ ]:
# categorical predictors
categorical_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
    'PaymentMethod'
]

# build the shared preprocessing steps used by all model classes.
def make_common_stages(include_scaler=False):
    # impute the missing TotalCharges values 
    imputer = Imputer(inputCols=['TotalCharges'], outputCols=['TotalCharges_imputed'])

    # index the categorical predictors
    indexer = StringIndexer(
        inputCols=categorical_cols,
        outputCols=[f'{col_name}_idx' for col_name in categorical_cols],
        handleInvalid='keep',
    )

    # expand the indexed categories into indicator vectors.
    encoder = OneHotEncoder(
        inputCols=[f'{col_name}_idx' for col_name in categorical_cols],
        outputCols=[f'{col_name}_oh' for col_name in categorical_cols],
        handleInvalid='keep',
    )

    # combine the encoded categorical pieces and the numeric columns into one vector.
    assembler_inputs = [f'{col_name}_oh' for col_name in categorical_cols] + [
        'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges_imputed'
    ]
    assembler = VectorAssembler(inputCols=assembler_inputs, outputCol='features_raw')

    # keep the shared preprocessing steps together and add scaling only when needed
    stages = [imputer, indexer, encoder, assembler]
    if include_scaler:
        # scale the final vector for logistic regression
        scaler = StandardScaler(inputCol='features_raw', outputCol='features', withMean=False, withStd=True)
        stages.append(scaler)

    return stages

# will use this AUC evaluator in cross-validation runs
evaluator = BinaryClassificationEvaluator(
    labelCol='label',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC',
)


lr = LogisticRegression(labelCol='label', featuresCol='features', maxIter=100)
# wrap the preprocessing steps and the logistic model into one pipeline
lr_pipeline = Pipeline(stages=make_common_stages(include_scaler=True) + [lr])
# build the regularization grid 
lr_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.0, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 1.0])
    .build()
)
# set up 3-fold CV for the logistic-regression pipeline
lr_cv = CrossValidator(
    estimator=lr_pipeline,
    estimatorParamMaps=lr_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,
    seed=RANDOM_STATE,
)

# fit logistic-regression tuning combinations
lr_cv_model = lr_cv.fit(train)
# find which tuning combination gave the largest mean CV AUC.
lr_best_idx = int(np.argmax(lr_cv_model.avgMetrics))
# pull the tuning values for that best logistic-regression fit.
lr_best_params = {param.name: value for param, value in lr_grid[lr_best_idx].items()}

# decision tree
dt = DecisionTreeClassifier(labelCol='label', featuresCol='features_raw', seed=RANDOM_STATE)

# wrap the shared preprocessing steps and the decision tree into pipeline.
dt_pipeline = Pipeline(stages=make_common_stages(include_scaler=False) + [dt])

# build the decision-tree tuning grid over depth and minimum node size.
dt_grid = (
    ParamGridBuilder()
    .addGrid(dt.maxDepth, [4, 7])
    .addGrid(dt.minInstancesPerNode, [20, 50])
    .build()
)

# set up 3-fold CV for the decision-tree pipeline.
dt_cv = CrossValidator(
    estimator=dt_pipeline,
    estimatorParamMaps=dt_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,
    seed=RANDOM_STATE,
)
# fit every decision-tree tuning combination on the training split.
dt_cv_model = dt_cv.fit(train)
# find which tuning combination gave the largest mean CV AUC
dt_best_idx = int(np.argmax(dt_cv_model.avgMetrics))
# pull the tuning values 
dt_best_params = {param.name: value for param, value in dt_grid[dt_best_idx].items()}


# random forest
rf = RandomForestClassifier(labelCol='label', featuresCol='features_raw', seed=RANDOM_STATE)

# wrap preprocessing steps and random forest into pipeline
rf_pipeline = Pipeline(stages=make_common_stages(include_scaler=False) + [rf])

# build the random-forest tuning grid over tree count and depth.
rf_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [100, 300])
    .addGrid(rf.maxDepth, [5, 8])
    .build()
)

# set up 3-fold CV 
rf_cv = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=rf_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,
    seed=RANDOM_STATE,
)

# fit random-forest 
rf_cv_model = rf_cv.fit(train)
# find which tuning combination gave the largest mean CV AUC
rf_best_idx = int(np.argmax(rf_cv_model.avgMetrics))
# pull the tuning values 
rf_best_params = {param.name: value for param, value in rf_grid[rf_best_idx].items()}

# collect the best CV results and tuning values in one dataframe.
training_summary = pd.DataFrame(
    [
        {
            'model': 'Logistic regression',
            'best_cv_auc': max(lr_cv_model.avgMetrics),
            'regParam': lr_best_params.get('regParam', np.nan),
            'elasticNetParam': lr_best_params.get('elasticNetParam', np.nan),
            'maxDepth': np.nan,
            'minInstancesPerNode': np.nan,
            'numTrees': np.nan,
        },
        {
            'model': 'Decision tree',
            'best_cv_auc': max(dt_cv_model.avgMetrics),
            'regParam': np.nan,
            'elasticNetParam': np.nan,
            'maxDepth': dt_best_params.get('maxDepth', np.nan),
            'minInstancesPerNode': dt_best_params.get('minInstancesPerNode', np.nan),
            'numTrees': np.nan,
        },
        {
            'model': 'Random forest',
            'best_cv_auc': max(rf_cv_model.avgMetrics),
            'regParam': np.nan,
            'elasticNetParam': np.nan,
            'maxDepth': rf_best_params.get('maxDepth', np.nan),
            'minInstancesPerNode': np.nan,
            'numTrees': rf_best_params.get('numTrees', np.nan),
        },
    ]
).sort_values('best_cv_auc', ascending=False).reset_index(drop=True)

# show the training-stage model comparison table.
display(training_summary)

,model,best_cv_auc,regParam,elasticNetParam,maxDepth,minInstancesPerNode,numTrees
0,Logistic regression,0.8436,0.0000,0.0000,NaN,NaN,NaN
1,Random forest,0.8428,NaN,NaN,8.0000,NaN,300.0000
2,Decision tree,0.6733,NaN,NaN,4.0000,20.0000,NaN


For training, logistic regression had the highest cross-validated AUC at `0.8436`, using `regParam = 0` and `elasticNetParam = 0`. The best random forest was very similar at `0.8428` with `300` trees and `maxDepth = 8`. The single decision tree was much smaller at `0.6733`.

## 4. Test the Models

Now I take the best model from each class and evaluate it on the test set with the AUC metric. 

In [ ]:
# list to store one test-set summary row per model.
test_rows = []
for model_name, cv_model in [
    ('Logistic regression', lr_cv_model),
    ('Decision tree', dt_cv_model),
    ('Random forest', rf_cv_model),
]:
    # apply the fitted cross-validated pipeline to the test Spark DataFrame
    test_predictions = cv_model.transform(test)

    # append the best training-stage CV AUC and the test-set AUC for this fitted model.
    test_rows.append(
        {
            'model': model_name,
            'best_cv_auc': float(training_summary.loc[training_summary['model'] == model_name, 'best_cv_auc'].iloc[0]),
            'test_auc': evaluator.evaluate(test_predictions),
        }
    )

# convert to pandas DataFrame, sort by descending test AUC
test_summary = pd.DataFrame(test_rows).sort_values('test_auc', ascending=False).reset_index(drop=True)



# display the final test-set model comparison table.
display(test_summary)


spark.stop()


,model,best_cv_auc,test_auc
0,Random forest,0.8428,0.8531
1,Logistic regression,0.8436,0.8493
2,Decision tree,0.6733,0.6777


The random forest finished with the best test AUC at `0.8531`, ahead of logistic regression at `0.8493` and the single decision tree at `0.6777`. This order matches the training results too: logistic regression and random forest were close to each other on cross-validated AUC, and both were ahead of the single tree.